In [1]:
# -----######-----###### FULL BATCH W/ CUSTOM CUT OPTION + TQDM -----######-----######
from moviepy.editor import VideoFileClip
from pathlib import Path
from tqdm import tqdm
import random

def _video_0506_storybatch_GET_all9_random_custom(input_folder, duration=60):
    input_folder = Path(input_folder)
    video_paths = [p for p in input_folder.iterdir() if p.suffix.lower() == '.mp4']

    if not video_paths:
        return "❌ No .mp4 files found in input folder."

    for video_path in video_paths:
        clip = VideoFileClip(str(video_path))
        total_duration = clip.duration
        usable_range = total_duration - duration

        if usable_range <= 0:
            print(f"⚠️ Skipping {video_path.name} (too short)")
            continue

        output_dir = input_folder / video_path.stem
        output_dir.mkdir(exist_ok=True)

        base_points = [usable_range * i / 9 for i in range(9)]
        random_offsets = [random.uniform(0, usable_range / 9) for _ in range(9)]
        start_times = [min(bp + ro, usable_range) for bp, ro in zip(base_points, random_offsets)]

        print(f"\n📤 Processing: {video_path.name}")
        for i, start_time in tqdm(enumerate(start_times), total=9, desc=f"⏳ {video_path.stem[:20]}"):
            subclip = clip.subclip(start_time, start_time + duration)
            subclip_resized = subclip.resize(height=1080).crop(x_center=subclip.w / 2, width=608)
            out_path = output_dir / f"{video_path.stem}_story_{i+1:02}.mp4"
            subclip_resized.write_videofile(str(out_path), codec="libx264", audio_codec="aac", verbose=False, logger=None)

        print(f"✅ 9 random clips saved for: {video_path.stem}")

        custom_index = 1
        while True:
            response = input(f"✂️  Export custom cut from {video_path.name}? (y/n): ").strip().lower()
            if response != 'y':
                break

            range_str = input("🕐 Enter start-end in minutes (e.g. 1.2 - 3.2): ").strip()
            try:
                start_min, end_min = [float(x.strip()) for x in range_str.split("-")]
                start_sec = start_min * 60
                end_sec = end_min * 60

                if end_sec > total_duration or start_sec >= end_sec:
                    print("❌ Invalid range. Try again.")
                    continue

                subclip = clip.subclip(start_sec, end_sec)
                subclip_resized = subclip.resize(height=1080).crop(x_center=subclip.w / 2, width=608)
                out_path = output_dir / f"__custom_{video_path.stem}_{custom_index}.mp4"
                subclip_resized.write_videofile(str(out_path), codec="libx264", audio_codec="aac", verbose=False, logger=None)
                print(f"✅ Custom clip saved: {out_path.name}")
                custom_index += 1

            except Exception as e:
                print(f"❌ Error parsing input: {e}")
                continue

    return "✅ All videos processed with optional custom cuts."


In [2]:
_video_0506_storybatch_GET_all9_random_custom(
    "/Users/yerik/Desktop/input_vids",
    duration=60
)



📤 Processing: vid_2025_05_17_Hr_23_18_36_Yfram_25_Yvres_1920x1080_Yaspr_16_9_Ydura_81_MIN_yeriko_no_1.mp4


OSError: MoviePy error: failed to read the first frame of video file /Users/yerik/Desktop/input_vids/vid_2025_05_17_Hr_23_18_36_Yfram_25_Yvres_1920x1080_Yaspr_16_9_Ydura_81_MIN_yeriko_no_1.mp4. That might mean that the file is corrupted. That may also mean that you are using a deprecated version of FFMPEG. On Ubuntu/Debian for instance the version in the repos is deprecated. Please update to a recent version from the website.